# 第 5 章｜Tool + Resource = AI Agent

依序執行每一格；可修改標示的參數後重跑。

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from course_utils import *
print("教材根目錄：", ROOT)
config = require_cillm_config()
print("執行模式：CILLM API（必要）")
print("GPT-OSS 模型：", config["model"])

> **執行模式 Hint**
>
> - 本教材每章都必須設定 `CILLM_API_KEY` 與 `CILLM_BASE_URL`，並呼叫 `openai/gpt-oss-120b`。
> - 缺少設定時會立即停止，不會以 mock 回覆取代真實模型。
> - 圖片解析另使用 `google/gemma-4-31b-it`，但沿用相同 CILLM API key。

> **Agent Hint**
>
> - 問純計算：預期只用 Tool。
> - 問公司規範：預期只讀 Resource。
> - 問「哪些 Excel 航班符合補償，以及提供什麼服務」：預期先讀規範，再執行 Excel Tool。
> - 問圖片中的班號是否符合延誤規範：可觀察圖片 Tool 加 Resource 的多步驟需求。

三個案例依序展示只用 Tool、只用 Resource，以及兩者並用。

In [ ]:
# 案例 1：只用 Tool
q1="312 個座位、載客率 87%，約有多少旅客？"; r1=round(math_tool(312,"*",.87))
print_execution_trace(question=q1, tool="✓ math_tool", resource="✗", tool_result=r1, answer=f"約 {r1} 位旅客。")

In [ ]:
# 案例 2：只用 Resource
q2="特殊餐點要多久前申請？"; rn=choose_resource(q2); ctx=search_resource(rn,q2)
print_execution_trace(question=q2, tool="✗", resource=f"✓ {rn}", answer="須於起飛前 24 小時申請。")

In [ ]:
# 案例 3：先查門檻，再用 Excel Tool 找航班
USER_REQUEST="哪些航班符合延誤補償條件，以及應該提供什麼服務？"
resource_name="passenger_service_rules"; rule=search_resource(resource_name,"延誤 補償 餐飲券")
threshold=120
code=f"df = pd.read_excel(excel_path)\nresult = df.loc[df['delay_minutes'] >= {threshold}, ['flight','delay_minutes']].to_dict('records')"
flights=safe_excel_python(ROOT / "data/excel/flight_delays.xlsx",code)
answer=ask_gpt_oss(USER_REQUEST, "規範：\n"+rule+"\n\nExcel Tool 結果：\n"+json.dumps(flights,ensure_ascii=False), "整合規範與 Tool 結果回答，不可加入未提供的資訊。使用繁體中文。")
print("步驟 1 Resource：\n",rule); print("步驟 2 Tool 結果："); show(flights)
print_execution_trace(question=USER_REQUEST, normalized="✓ Excel 資料", tool="✓ excel_python_tool", resource="✓ passenger_service_rules", tool_result=flights, answer=answer)

### 小練習

修改 `USER_REQUEST` 或補償門檻，觀察 Agent 所需能力。